# Eigensolutions and Solving $A \mathbf{x} = b$

## Problem Statement
Given the $2 \times 2$ symmetric system:
$$A \mathbf{x} = b$$

where:
$$A = \begin{bmatrix} 0 & 1 \\ 1 & -2 \end{bmatrix}, \quad b = \begin{bmatrix} 3 \\ 4 \end{bmatrix}, \quad \mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}$$

We want to:
1. Use the determinant condition $\det(A - kI) = 0$ to find the scalar eigenvalues $k$.
2. Find the corresponding eigenvectors $\mathbf{x}_0, \mathbf{x}_1$.
3. Use eigendecomposition ($A = Q K Q^T$) to decouple and solve $A \mathbf{x} = b$.

---

## Theoretical Foundations (Lecture Slide 03: Slides 31–35)

### 1. Why Do We Set $\det(A - kI) = 0$?
An eigenvector $\mathbf{x}$ and its eigenvalue $k$ satisfy:
$$A \mathbf{x} = k \mathbf{x} \implies A \mathbf{x} - k I \mathbf{x} = \mathbf{0} \implies (A - k I)\mathbf{x} = \mathbf{0}$$

* If $(A - k I)$ had a non-zero determinant ($\det(A - k I) \neq 0$), then $(A - k I)^{-1}$ would exist.
* Multiplying both sides by the inverse would force:
  $$\mathbf{x} = (A - k I)^{-1} \mathbf{0} = \mathbf{0}$$
  which is the trivial solution ($\mathbf{x} = \mathbf{0}$ carries no directional information).
* Therefore, to find **non-trivial (non-zero)** vectors $\mathbf{x} \neq \mathbf{0}$, the matrix $(A - k I)$ **must be singular (non-invertible)**:
  $$\det(A - k I) = 0$$

### 2. Difference Between Rotations and Eigendecomposition
* **Givens / Householder:** Transform $A$ into an upper-triangular matrix $R$ using external geometric reflections/rotations to perform row elimination.
* **Eigendecomposition:** Discovers the **intrinsic natural axes** (\"skeleton\") of $A$ where cross-talk is zero ($A \mathbf{x}_i = k_i \mathbf{x}_i$). Along these axes, matrix multiplication collapses into pure scalar scaling by $k_i$.


In [1]:
import numpy as np

# Define system A x = b
A = np.array([
    [0.0,  1.0],
    [1.0, -2.0]
])

b = np.array([3.0, 4.0])

print('Matrix A:')
print(A)
print('\nRHS Vector b:')
print(b)
print(f'\nDeterminant of A: det(A) = {np.linalg.det(A):.1f}')


Matrix A:
[[ 0.  1.]
 [ 1. -2.]]

RHS Vector b:
[3. 4.]

Determinant of A: det(A) = -1.0


## Step 1: Finding the Eigenvalues $k$ via $\det(A - kI) = 0$

Construct the matrix $(A - k I)$:
$$A - k I = \begin{bmatrix} 0 - k & 1 \\ 1 & -2 - k \end{bmatrix} = \begin{bmatrix} -k & 1 \\ 1 & -(2 + k) \end{bmatrix}$$

From Slide 34, the determinant of a $2 \times 2$ matrix $\begin{bmatrix} a & b \\ c & d \end{bmatrix}$ is $ad - bc$:
$$\det(A - k I) = (-k)(-2 - k) - (1)(1) = k(k + 2) - 1 = k^2 + 2k - 1 = 0$$

Using the quadratic formula $k = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$:
$$k = \frac{-2 \pm \sqrt{2^2 - 4(1)(-1)}}{2(1)} = \frac{-2 \pm \sqrt{4 + 4}}{2} = \frac{-2 \pm \sqrt{8}}{2} = -1 \pm \sqrt{2}$$

The two scalar eigenvalues are:
* $k_0 = -1 + \sqrt{2} \approx 0.41421356$
* $k_1 = -1 - \sqrt{2} \approx -2.41421356$

> **Solvability Check (Slide 31):** $\det(A) = k_0 \cdot k_1 = (-1 + \sqrt{2})(-1 - \sqrt{2}) = 1 - 2 = -1 \neq 0$.
> Since neither eigenvalue is 0, the determinant is non-zero, guaranteeing a unique solution to $A \mathbf{x} = b$.


In [2]:
# Characteristic polynomial coefficients for k^2 + 2k - 1 = 0
# a*k^2 + b*k + c = 0
a_coeff = 1.0
b_coeff = -(A[0, 0] + A[1, 1])          # -trace(A) = -(-2) = 2
c_coeff = A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0] # det(A) = -1

disc = np.sqrt(b_coeff**2 - 4 * a_coeff * c_coeff) # sqrt(4 - 4(-1)) = sqrt(8)
k0 = (-b_coeff + disc) / (2 * a_coeff) # -1 + sqrt(2)
k1 = (-b_coeff - disc) / (2 * a_coeff) # -1 - sqrt(2)

print(f'Characteristic equation: k^2 + ({b_coeff:.1f})k + ({c_coeff:.1f}) = 0')
print(f'Eigenvalue k0: -1 + sqrt(2) = {k0:.8f}')
print(f'Eigenvalue k1: -1 - sqrt(2) = {k1:.8f}')

# Check det(A) = k0 * k1
print(f'Product k0 * k1: {k0 * k1:.4f} (matches det(A) = {np.linalg.det(A):.4f})')


Characteristic equation: k^2 + (2.0)k + (-1.0) = 0
Eigenvalue k0: -1 + sqrt(2) = 0.41421356
Eigenvalue k1: -1 - sqrt(2) = -2.41421356
Product k0 * k1: -1.0000 (matches det(A) = -1.0000)


## Step 2: Finding the Eigenvectors $\mathbf{x}_0$ and $\mathbf{x}_1$

For each eigenvalue $k_i$, solve the homogeneous system $(A - k_i I)\mathbf{x} = \mathbf{0}$:
$$\begin{bmatrix} -k_i & 1 \\ 1 & -2 - k_i \end{bmatrix} \begin{bmatrix} x_{i, 0} \\ x_{i, 1} \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$$

From row 0:
$$-k_i x_{i, 0} + x_{i, 1} = 0 \implies x_{i, 1} = k_i x_{i, 0}$$

Following the lecture convention (Slide 35: *\"We can always choose the 1st element, $x_{i, 0}$, to be 1 leaving...\"*):
$$\mathbf{x}_i = \begin{bmatrix} 1 \\ k_i \end{bmatrix}$$

1. **For $k_0 = -1 + \sqrt{2} \approx 0.4142$:**
   $$\mathbf{x}_0 = \begin{bmatrix} 1 \\ -1 + \sqrt{2} \end{bmatrix} \approx \begin{bmatrix} 1 \\ 0.4142 \end{bmatrix}$$
2. **For $k_1 = -1 - \sqrt{2} \approx -2.4142$:**
   $$\mathbf{x}_1 = \begin{bmatrix} 1 \\ -1 - \sqrt{2} \end{bmatrix} \approx \begin{bmatrix} 1 \\ -2.4142 \end{bmatrix}$$

Notice that because $A$ is symmetric, the eigenvectors are orthogonal:
$$\mathbf{x}_0^T \mathbf{x}_1 = 1(1) + (-1 + \sqrt{2})(-1 - \sqrt{2}) = 1 + (1 - 2) = 0$$


In [3]:
# Form unnormalized eigenvectors using Slide 35 convention (x_0 = 1)
x_vec0 = np.array([1.0, k0])
x_vec1 = np.array([1.0, k1])

print('Eigenvector x0 (associated with k0):')
print(x_vec0)
print('\nEigenvector x1 (associated with k1):')
print(x_vec1)

# Verification: A @ x = k * x
print('\nVerification of A @ x = k * x:')
print('A @ x0:', A @ x_vec0)
print('k0 * x0:', k0 * x_vec0)
print('Difference ||A x0 - k0 x0||:', np.linalg.norm(A @ x_vec0 - k0 * x_vec0))

print('\nA @ x1:', A @ x_vec1)
print('k1 * x1:', k1 * x_vec1)
print('Difference ||A x1 - k1 x1||:', np.linalg.norm(A @ x_vec1 - k1 * x_vec1))

# Check orthogonality: x0.T @ x1 == 0
print('\nOrthogonality check (x0 . x1):', np.dot(x_vec0, x_vec1))


Eigenvector x0 (associated with k0):
[1.         0.41421356]

Eigenvector x1 (associated with k1):
[ 1.         -2.41421356]

Verification of A @ x = k * x:
A @ x0: [0.41421356 0.17157288]
k0 * x0: [0.41421356 0.17157288]
Difference ||A x0 - k0 x0||: 2.7755575615628914e-16

A @ x1: [-2.41421356  5.82842712]
k1 * x1: [-2.41421356  5.82842712]
Difference ||A x1 - k1 x1||: 0.0

Orthogonality check (x0 . x1): -2.220446049250313e-16


## Step 3: Solving $A \mathbf{x} = b$ via Eigendecomposition

Because $A$ is symmetric, its eigenvectors form an orthogonal basis.
Normalizing them to unit length produces an **orthonormal matrix** $Q = [\mathbf{q}_0, \mathbf{q}_1]$ where $Q^T Q = I$:
$$\mathbf{q}_0 = \frac{\mathbf{x}_0}{\|\mathbf{x}_0\|}, \quad \mathbf{q}_1 = \frac{\mathbf{x}_1}{\|\mathbf{x}_1\|}$$

The matrix $A$ decomposes into its diagonal eigenvalue skeleton (Slides 37–40):
$$A = Q K Q^T = Q \begin{bmatrix} k_0 & 0 \\ 0 & k_1 \end{bmatrix} Q^T$$

To solve $A \mathbf{x} = b$:
$$Q K Q^T \mathbf{x} = b \implies K (Q^T \mathbf{x}) = Q^T b$$

1. **Switch to Modal Frame:** $c = Q^T b$ (projects RHS $b$ onto each eigen-axis).
2. **Independent Scaling:** Divide channel $i$ by its eigenvalue $k_i$ (no cross-talk!):
   $$y_i = \frac{c_i}{k_i}$$
3. **Reconstruct Physical Solution:**
   $$\mathbf{x} = Q y = \sum_{i=0}^1 \frac{\mathbf{q}_i^T b}{k_i} \mathbf{q}_i$$


In [4]:
# Normalize eigenvectors to form orthonormal matrix Q
q0 = x_vec0 / np.linalg.norm(x_vec0)
q1 = x_vec1 / np.linalg.norm(x_vec1)
Q = np.column_stack([q0, q1])
K = np.diag([k0, k1])

print('Orthonormal Eigenvector Matrix Q:')
print(np.round(Q, 4))
print(f'Q is orthogonal (Q.T @ Q == I): {np.allclose(Q.T @ Q, np.eye(2))}')

# Reconstruct A = Q @ K @ Q.T
A_reconstructed = Q @ K @ Q.T
print('\nReconstruction Q @ K @ Q.T:')
print(np.round(A_reconstructed, 4))

# Step 1: Project b onto eigenbasis: c = Q^T @ b
c = Q.T @ b
# Step 2: Uncoupled division: y = c / k
y = c / np.array([k0, k1])
# Step 3: Reconstruct x in physical space: x = Q @ y
x_sol = Q @ y

print('\nComputed Solution x = [x1, x2]^T via Eigendecomposition:')
print(x_sol)

# Direct algebraic check:
# Row 0: 0*x1 + 1*x2 = 3   => x2 = 3
# Row 1: 1*x1 - 2*x2 = 4   => x1 - 2(3) = 4 => x1 = 10
print('\nDirect Algebraic Solution: [10, 3]')

# Residual norm
residual = A @ x_sol - b
print('A @ x_sol:', A @ x_sol)
print('Vector b: ', b)
print('Residual norm ||A x - b||:', np.linalg.norm(residual))


Orthonormal Eigenvector Matrix Q:
[[ 0.9239  0.3827]
 [ 0.3827 -0.9239]]
Q is orthogonal (Q.T @ Q == I): True

Reconstruction Q @ K @ Q.T:
[[ 0.  1.]
 [ 1. -2.]]

Computed Solution x = [x1, x2]^T via Eigendecomposition:
[10.  3.]

Direct Algebraic Solution: [10, 3]
A @ x_sol: [3. 4.]
Vector b:  [3. 4.]
Residual norm ||A x - b||: 2.7012892057857038e-15


---

## Summary: Eigendecomposition vs. Elimination (Givens / Householder)

| Method | Primary Operation | Transformation Type | Key Advantage |
| :--- | :--- | :--- | :--- |
| **Eigendecomposition** ($A = Q K Q^T$) | Find roots of $\det(A - kI) = 0$ | Change to natural uncoupled coordinates | Solves powers $A^M$, exponentials $e^{At}$, and decouples physical modes |
| **Householder Reflection** ($A = Q R$) | Reflect column $u = x - x'$ | Orthogonal triangularization | Highly efficient and numerically stable for dense linear systems $A\mathbf{x} = b$ |
| **Givens Rotation** ($A = Q R$) | Planar rotation by $\theta$ | Pairwise coordinate rotation | Selective zeroing for sparse or banded matrices |
